In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)


In [5]:
# ================================================================
# TRUE DIC — One-Step Conditional Likelihood
# IID + Weekly BYM + Weekly BYM + Cov
# 200 posterior samples
# ================================================================

import numpy as np
import geopandas as gpd
import pyreadr
import pickle
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from tqdm import tqdm
from pathlib import Path

# ------------------------------------------------
# Config
# ------------------------------------------------
DIST_TH = 0.22
period = 52
BASE_DIR = Path(r"D:\77\Research\temp\snow")
N_SAMPLE = 1000

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

# ================================================================
# 1️⃣ Load & filter data
# ================================================================

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords_all = snow.iloc[:, :2].to_numpy()
y_all = snow.iloc[:, 2:].to_numpy()

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_all[:,0], coords_all[:,1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
Dmat = squareform(pdist(xy))

W = (Dmat <= DIST_TH).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

n_comp, labels = connected_components(W, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

use_idx = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

coords = coords_all[use_idx]
y = y_all[use_idx]
S, TT = y.shape
print("Using S =", S, "TT =", TT)

# ------------------------------------------------
# Time covariates
# ------------------------------------------------
t_full = np.arange(1, TT + 1)
t_scaled = (t_full - t_full.mean()) / t_full.std(ddof=0)

cos_all = np.cos(2*np.pi*np.arange(1,TT)/period)
sin_all = np.sin(2*np.pi*np.arange(1,TT)/period)
trend_all = t_scaled[:-1]

# ------------------------------------------------
# Covariates for +Cov model
# ------------------------------------------------
snow_temp = pyreadr.read_r(BASE_DIR/"snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)
temp_full = snow_temp.drop(index=no_nbs).iloc[:,2:].to_numpy()
temp_full = temp_full[use_idx]
temp_scaled = (temp_full - temp_full.mean()) / temp_full.std()

lat = coords[:,1]
lat = (lat - lat.mean()) / lat.std()

elev_raw = pd.read_csv(BASE_DIR/"curr_elev.csv").iloc[:,3].to_numpy()
elev = (elev_raw[use_idx] - elev_raw[use_idx].mean()) / elev_raw[use_idx].std()

# ================================================================
# Helper: one-step likelihood
# ================================================================

def one_step_ll(p01, p10):

    ll = 0.0
    for t in range(TT-1):

        y_prev = y[:,t]
        y_next = y[:,t+1]

        prob = np.where(
            y_prev==0,
            np.where(y_next==1, p01[t], 1-p01[t]),
            np.where(y_next==0, p10[t], 1-p10[t])
        )

        ll += np.sum(np.log(prob + 1e-12))

    return ll

# ================================================================
# 1️⃣ IID
# ================================================================

print("\n===== IID TRUE DIC =====")

theta01 = np.load(BASE_DIR/"ind01.npz")["all_theta"][:, :N_SAMPLE]
theta10 = np.load(BASE_DIR/"ind10.npz")["all_theta"][:, :N_SAMPLE]

def compute_iid_ll(th01, th10):

    p01_list = []
    p10_list = []

    for t in range(TT-1):

        cos_t = cos_all[t]
        sin_t = sin_all[t]
        trend = trend_all[t]

        phi01 = th01[0] + th01[1]*cos_t + th01[2]*sin_t + th01[3]*trend
        phi10 = th10[0] + th10[1]*cos_t + th10[2]*sin_t + th10[3]*trend

        p01_list.append(1/(1+np.exp(-phi01)))
        p10_list.append(1/(1+np.exp(-phi10)))

    return one_step_ll(p01_list, p10_list)

D_vals = np.zeros(N_SAMPLE)

for m in tqdm(range(N_SAMPLE)):
    D_vals[m] = -2*compute_iid_ll(
        theta01[:,m].reshape(4,S),
        theta10[:,m].reshape(4,S)
    )

D_bar = D_vals.mean()
D_hat = -2*compute_iid_ll(
    theta01.mean(axis=1).reshape(4,S),
    theta10.mean(axis=1).reshape(4,S)
)

DIC_IID = 2*D_bar - D_hat
print("IID DIC :", DIC_IID)

# ================================================================
# 2️⃣ Weekly BYM
# ================================================================

print("\n===== Weekly BYM TRUE DIC =====")

with open(BASE_DIR/"bym01_weekly.pkl","rb") as f:
    res01 = pickle.load(f)
with open(BASE_DIR/"bym10_weekly.pkl","rb") as f:
    res10 = pickle.load(f)

eta01 = res01["all_eta"][:, :N_SAMPLE]
tau01 = res01["all_tau"][:, :N_SAMPLE]
eta10 = res10["all_eta"][:, :N_SAMPLE]
tau10 = res10["all_tau"][:, :N_SAMPLE]

K_total = 8

def compute_weekly_ll(e01,t01,e10,t10):

    e01_sp = e01.reshape(K_total,S)
    e10_sp = e10.reshape(K_total,S)

    t01_mat = t01.reshape(K_total,period)
    t10_mat = t10.reshape(K_total,period)

    p01_list = []
    p10_list = []

    for t in range(TT-1):

        week = t % period
        cos_t = cos_all[t]
        sin_t = sin_all[t]
        trend = trend_all[t]

        cov_vec = np.array([1,1,cos_t,cos_t,sin_t,sin_t,trend,trend])

        phi01 = np.sum(cov_vec[:,None] * e01_sp * t01_mat[:,week][:,None], axis=0)
        phi10 = np.sum(cov_vec[:,None] * e10_sp * t10_mat[:,week][:,None], axis=0)

        p01_list.append(1/(1+np.exp(-phi01)))
        p10_list.append(1/(1+np.exp(-phi10)))

    return one_step_ll(p01_list, p10_list)

D_vals = np.zeros(N_SAMPLE)

for m in tqdm(range(N_SAMPLE)):
    D_vals[m] = -2*compute_weekly_ll(
        eta01[:,m], tau01[:,m],
        eta10[:,m], tau10[:,m]
    )

D_bar = D_vals.mean()
D_hat = -2*compute_weekly_ll(
    eta01.mean(axis=1), tau01.mean(axis=1),
    eta10.mean(axis=1), tau10.mean(axis=1)
)

DIC_weekly = 2*D_bar - D_hat
print("Weekly DIC :", DIC_weekly)

# ================================================================
# 3️⃣ Weekly BYM + Cov
# ================================================================

print("\n===== Weekly BYM + Cov TRUE DIC =====")

with open(BASE_DIR/"bym01_cov_weekly_tau9.pkl","rb") as f:
    res01 = pickle.load(f)
with open(BASE_DIR/"bym10_cov_weekly_tau9.pkl","rb") as f:
    res10 = pickle.load(f)

eta01 = res01["all_eta"][:, :N_SAMPLE]
tau01 = res01["all_tau"][:, :N_SAMPLE]
eta10 = res10["all_eta"][:, :N_SAMPLE]
tau10 = res10["all_tau"][:, :N_SAMPLE]

def compute_weekly_cov_ll(e01,t01,e10,t10):

    e01_sp = e01[:K_total*S].reshape(K_total,S)
    gamma01 = e01[K_total*S:]

    e10_sp = e10[:K_total*S].reshape(K_total,S)
    gamma10 = e10[K_total*S:]

    t01_mat = t01.reshape(K_total,period)
    t10_mat = t10.reshape(K_total,period)

    p01_list = []
    p10_list = []

    for t in range(TT-1):

        week = t % period
        cos_t = cos_all[t]
        sin_t = sin_all[t]
        trend = trend_all[t]

        cov_vec = np.array([1,1,cos_t,cos_t,sin_t,sin_t,trend,trend])

        phi01_sp = np.sum(cov_vec[:,None] * e01_sp * t01_mat[:,week][:,None], axis=0)
        phi10_sp = np.sum(cov_vec[:,None] * e10_sp * t10_mat[:,week][:,None], axis=0)

        phi01 = phi01_sp + trend*lat*gamma01[0] + trend*elev*gamma01[1] + trend*temp_scaled[:,t]*gamma01[2]
        phi10 = phi10_sp + trend*lat*gamma10[0] + trend*elev*gamma10[1] + trend*temp_scaled[:,t]*gamma10[2]

        p01_list.append(1/(1+np.exp(-phi01)))
        p10_list.append(1/(1+np.exp(-phi10)))

    return one_step_ll(p01_list, p10_list)

D_vals = np.zeros(N_SAMPLE)

for m in tqdm(range(N_SAMPLE)):
    D_vals[m] = -2*compute_weekly_cov_ll(
        eta01[:,m], tau01[:,m],
        eta10[:,m], tau10[:,m]
    )

D_bar = D_vals.mean()
D_hat = -2*compute_weekly_cov_ll(
    eta01.mean(axis=1), tau01.mean(axis=1),
    eta10.mean(axis=1), tau10.mean(axis=1)
)

DIC_weekly_cov = 2*D_bar - D_hat
print("Weekly + Cov DIC :", DIC_weekly_cov)

# ================================================================
print("\n==============================")
print("IID DIC              :", DIC_IID)
print("Weekly BYM DIC       :", DIC_weekly)
print("Weekly BYM + Cov DIC :", DIC_weekly_cov)
print("==============================")

Using S = 1557 TT = 2704

===== IID TRUE DIC =====


100%|██████████| 1000/1000 [04:40<00:00,  3.57it/s]


IID DIC : 1234622.6568188646

===== Weekly BYM TRUE DIC =====


100%|██████████| 1000/1000 [08:27<00:00,  1.97it/s]


Weekly DIC : 1192755.7197864228

===== Weekly BYM + Cov TRUE DIC =====


100%|██████████| 1000/1000 [10:12<00:00,  1.63it/s]


Weekly + Cov DIC : 1194642.1621443478

IID DIC              : 1234622.6568188646
Weekly BYM DIC       : 1192755.7197864228
Weekly BYM + Cov DIC : 1194642.1621443478


In [6]:
# ================================================================
# SAFE WAIC (NO MEMORY EXPLOSION)
# IID / Weekly BYM / Weekly BYM + Cov
# Two largest connected components
# ================================================================

import numpy as np
import geopandas as gpd
import pyreadr
import pickle
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy.special import logsumexp
from pathlib import Path
from tqdm import tqdm

# ------------------------------------------------
# Config
# ------------------------------------------------
DIST_TH = 0.22
period = 52
BASE_DIR = Path(r"D:\77\Research\temp\snow")
N_SAMPLE = 1000

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1


# ================================================================
# 1️⃣ Load & filter data
# ================================================================
print("Loading data...")

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords_all = snow.iloc[:, :2].to_numpy()
y_all = snow.iloc[:, 2:].to_numpy()

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_all[:,0], coords_all[:,1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
Dmat = squareform(pdist(xy))

W = (Dmat <= DIST_TH).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

n_comp, labels = connected_components(W, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

use_idx = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

coords = coords_all[use_idx]
y = y_all[use_idx]
S, TT = y.shape

print("Using S =", S, "TT =", TT)


# ================================================================
# Time covariates
# ================================================================
t_full = np.arange(1, TT + 1)
t_scaled = (t_full - t_full.mean()) / t_full.std(ddof=0)

cos_all = np.cos(2*np.pi*np.arange(1,TT)/period)
sin_all = np.sin(2*np.pi*np.arange(1,TT)/period)
trend_all = t_scaled[:-1]


# ================================================================
# WAIC online accumulator
# ================================================================
def waic_update(loglik, lppd_acc, p_acc):
    lppd_acc += np.sum(
        logsumexp(loglik, axis=1) - np.log(N_SAMPLE)
    )
    p_acc += np.sum(
        np.var(loglik, axis=1)
    )
    return lppd_acc, p_acc


# ================================================================
# 1️⃣ IID
# ================================================================
print("\n===== IID WAIC =====")

theta01 = np.load(BASE_DIR/"ind01.npz")["all_theta"][:, :N_SAMPLE]
theta10 = np.load(BASE_DIR/"ind10.npz")["all_theta"][:, :N_SAMPLE]

theta01 = theta01.reshape(4, S, N_SAMPLE)
theta10 = theta10.reshape(4, S, N_SAMPLE)

lppd = 0.0
p_waic = 0.0

for t in tqdm(range(TT-1)):

    cos_t = cos_all[t]
    sin_t = sin_all[t]
    trend = trend_all[t]

    phi01 = (
        theta01[0]
        + theta01[1]*cos_t
        + theta01[2]*sin_t
        + theta01[3]*trend
    )

    phi10 = (
        theta10[0]
        + theta10[1]*cos_t
        + theta10[2]*sin_t
        + theta10[3]*trend
    )

    p01 = 1/(1+np.exp(-phi01))
    p10 = 1/(1+np.exp(-phi10))

    y_prev = y[:,t][:,None]
    y_next = y[:,t+1][:,None]

    prob = np.where(
        y_prev==0,
        np.where(y_next==1, p01, 1-p01),
        np.where(y_next==0, p10, 1-p10)
    )

    loglik = np.log(prob + 1e-12)

    lppd, p_waic = waic_update(loglik, lppd, p_waic)

WAIC_IID = -2 * (lppd - p_waic)

print("IID WAIC :", WAIC_IID)


# ================================================================
# 2️⃣ Weekly BYM
# ================================================================
print("\n===== Weekly BYM WAIC =====")

with open(BASE_DIR/"bym01_weekly.pkl","rb") as f:
    res01 = pickle.load(f)
with open(BASE_DIR/"bym10_weekly.pkl","rb") as f:
    res10 = pickle.load(f)

eta01 = res01["all_eta"][:, :N_SAMPLE]
tau01 = res01["all_tau"][:, :N_SAMPLE]
eta10 = res10["all_eta"][:, :N_SAMPLE]
tau10 = res10["all_tau"][:, :N_SAMPLE]

K_total = 8

eta01 = eta01.reshape(K_total, S, N_SAMPLE)
eta10 = eta10.reshape(K_total, S, N_SAMPLE)
tau01 = tau01.reshape(K_total, period, N_SAMPLE)
tau10 = tau10.reshape(K_total, period, N_SAMPLE)

lppd = 0.0
p_waic = 0.0

for t in tqdm(range(TT-1)):

    week = t % period
    cos_t = cos_all[t]
    sin_t = sin_all[t]
    trend = trend_all[t]

    cov_vec = np.array([1,1,cos_t,cos_t,sin_t,sin_t,trend,trend])

    phi01 = np.sum(
        cov_vec[:,None,None] *
        eta01 *
        tau01[:,week][:,None,:],
        axis=0
    )

    phi10 = np.sum(
        cov_vec[:,None,None] *
        eta10 *
        tau10[:,week][:,None,:],
        axis=0
    )

    p01 = 1/(1+np.exp(-phi01))
    p10 = 1/(1+np.exp(-phi10))

    y_prev = y[:,t][:,None]
    y_next = y[:,t+1][:,None]

    prob = np.where(
        y_prev==0,
        np.where(y_next==1, p01, 1-p01),
        np.where(y_next==0, p10, 1-p10)
    )

    loglik = np.log(prob + 1e-12)

    lppd, p_waic = waic_update(loglik, lppd, p_waic)

WAIC_weekly = -2 * (lppd - p_waic)

print("Weekly WAIC :", WAIC_weekly)


# ================================================================
# 3️⃣ Weekly BYM + Cov
# ================================================================
print("\n===== Weekly BYM + Cov WAIC =====")

with open(BASE_DIR/"bym01_cov_weekly_tau9.pkl","rb") as f:
    res01 = pickle.load(f)
with open(BASE_DIR/"bym10_cov_weekly_tau9.pkl","rb") as f:
    res10 = pickle.load(f)

eta01_full = res01["all_eta"][:, :N_SAMPLE]
tau01 = res01["all_tau"][:, :N_SAMPLE]
eta10_full = res10["all_eta"][:, :N_SAMPLE]
tau10 = res10["all_tau"][:, :N_SAMPLE]

gamma01 = eta01_full[K_total*S:, :]
gamma10 = eta10_full[K_total*S:, :]

eta01 = eta01_full[:K_total*S].reshape(K_total, S, N_SAMPLE)
eta10 = eta10_full[:K_total*S].reshape(K_total, S, N_SAMPLE)

tau01 = tau01.reshape(K_total, period, N_SAMPLE)
tau10 = tau10.reshape(K_total, period, N_SAMPLE)

# ---- load covariates ----
snow_temp = pyreadr.read_r(BASE_DIR/"snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)
temp_full = snow_temp.drop(index=no_nbs).iloc[:,2:].to_numpy()
temp_full = temp_full[use_idx]
temp_scaled = (temp_full - temp_full.mean()) / temp_full.std()

lat = coords[:,1]
lat = (lat - lat.mean()) / lat.std()

elev_raw = pd.read_csv(BASE_DIR/"curr_elev.csv").iloc[:,3].to_numpy()
elev = elev_raw[use_idx]
elev = (elev - elev.mean()) / elev.std()

lppd = 0.0
p_waic = 0.0

for t in tqdm(range(TT-1)):

    week = t % period
    cos_t = cos_all[t]
    sin_t = sin_all[t]
    trend = trend_all[t]

    cov_vec = np.array([1,1,cos_t,cos_t,sin_t,sin_t,trend,trend])

    phi01_sp = np.sum(
        cov_vec[:,None,None] *
        eta01 *
        tau01[:,week][:,None,:],
        axis=0
    )

    phi10_sp = np.sum(
        cov_vec[:,None,None] *
        eta10 *
        tau10[:,week][:,None,:],
        axis=0
    )

    phi01 = (
        phi01_sp
        + trend*lat[:,None]*gamma01[0]
        + trend*elev[:,None]*gamma01[1]
        + trend*temp_scaled[:,t][:,None]*gamma01[2]
    )

    phi10 = (
        phi10_sp
        + trend*lat[:,None]*gamma10[0]
        + trend*elev[:,None]*gamma10[1]
        + trend*temp_scaled[:,t][:,None]*gamma10[2]
    )

    p01 = 1/(1+np.exp(-phi01))
    p10 = 1/(1+np.exp(-phi10))

    y_prev = y[:,t][:,None]
    y_next = y[:,t+1][:,None]

    prob = np.where(
        y_prev==0,
        np.where(y_next==1, p01, 1-p01),
        np.where(y_next==0, p10, 1-p10)
    )

    loglik = np.log(prob + 1e-12)

    lppd, p_waic = waic_update(loglik, lppd, p_waic)

WAIC_weekly_cov = -2 * (lppd - p_waic)

print("Weekly + Cov WAIC :", WAIC_weekly_cov)


# ================================================================
# Final
# ================================================================
print("\n==============================")
print("IID WAIC              :", WAIC_IID)
print("Weekly BYM WAIC       :", WAIC_weekly)
print("Weekly BYM + Cov WAIC :", WAIC_weekly_cov)
print("==============================")

Loading data...
Using S = 1557 TT = 2704

===== IID WAIC =====


100%|██████████| 2703/2703 [06:22<00:00,  7.07it/s]


IID WAIC : 1235546.098987285

===== Weekly BYM WAIC =====


100%|██████████| 2703/2703 [10:26<00:00,  4.31it/s]


Weekly WAIC : 1198660.77498602

===== Weekly BYM + Cov WAIC =====


100%|██████████| 2703/2703 [12:16<00:00,  3.67it/s]

Weekly + Cov WAIC : 1197781.4966380775

IID WAIC              : 1235546.098987285
Weekly BYM WAIC       : 1198660.77498602
Weekly BYM + Cov WAIC : 1197781.4966380775
